# Retos · Juegos y simulaciones

Pequeños juegos y simulaciones por turnos: mover piezas en un tablero, adivinar códigos con pistas, combates y torneos, carreras, un clima inventado y un cuestionario. Para que el resultado se pueda reproducir, el azar sale siempre de un `random.Random` con semilla fija y las jugadas de las personas se simulan con listas. Cada ejercicio comprueba con `assert` los invariantes del juego (límites, totales, quién puede ganar). Ideas de [retosdeprogramacion.com](https://retosdeprogramacion.com/ejercicios).

## 1. Una pieza en un tablero de bloques

En un tablero de 10 × 10, una pieza en forma de L (cuatro casillas) cae desde arriba. Escribir `mover(pieza, accion)` para las acciones `izquierda`, `derecha`, `abajo` y `rotar` (90° en sentido horario alrededor de su segunda casilla), que devuelva la pieza nueva o la misma si el movimiento la saca del tablero. Aplicar una lista de acciones, avisar de las bloqueadas y dibujar el tablero. Comprobar que la pieza nunca sale, que girar conserva las distancias entre casillas y que cuatro giros la dejan como estaba.

In [1]:
import random
from itertools import combinations

LADO = 10
DESPLAZAMIENTOS = {"izquierda": (0, -1), "derecha": (0, 1), "abajo": (1, 0)}


def dentro(pieza):
    return all(0 <= f < LADO and 0 <= c < LADO for f, c in pieza)


def mover(pieza, accion):
    if accion == "rotar":
        pf, pc = pieza[1]
        nueva = [(pf + (c - pc), pc - (f - pf)) for f, c in pieza]
    else:
        df, dc = DESPLAZAMIENTOS[accion]
        nueva = [(f + df, c + dc) for f, c in pieza]
    return nueva if dentro(nueva) else pieza


def dibujar(pieza):
    for f in range(LADO):
        print(" ".join("■" if (f, c) in pieza else "·" for c in range(LADO)))


def distancias(pieza):
    return sorted((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2 for a, b in combinations(pieza, 2))


pieza = [(0, 4), (1, 4), (2, 4), (2, 5)]
acciones = ["abajo", "rotar", "derecha", "derecha", "derecha", "derecha", "derecha", "rotar", "rotar"] + ["abajo"] * 8
for n, accion in enumerate(acciones, 1):
    nueva = mover(pieza, accion)
    if nueva == pieza:
        print(f"Acción {n} ({accion}) bloqueada por el borde")
    assert dentro(nueva) and distancias(nueva) == distancias(pieza)
    pieza = nueva
dibujar(pieza)

centro = [(4, 4), (5, 4), (6, 4), (6, 5)]
girada = centro
for _ in range(4):
    girada = mover(girada, "rotar")
assert girada == centro
assert max(c for _, c in pieza) == LADO - 1 and max(f for f, _ in pieza) == LADO - 1

Acción 7 (derecha) bloqueada por el borde
Acción 17 (abajo) bloqueada por el borde
· · · · · · · · · ·
· · · · · · · · · ·
· · · · · · · · · ·
· · · · · · · · · ·
· · · · · · · · · ·
· · · · · · · · · ·
· · · · · · · · · ·
· · · · · · · · · ·
· · · · · · · · · ■
· · · · · · · ■ ■ ■


## 2. Descifrar el código

Un código secreto tiene 4 símbolos distintos elegidos entre `A`, `B`, `C`, `1`, `2` y `3`. En cada intento se dan pistas por posición: `correcto` (símbolo y posición), `presente` (está en otra posición) o `ausente`. Escribir `pistas(secreto, intento)` y `partida(secreto, intentos)`, que juegue como mucho 10 intentos válidos con una lista fija de propuestas, avise de las mal formadas sin gastar intento y diga si se ha acertado.

In [2]:
from itertools import permutations

SIMBOLOS = "ABC123"
LONGITUD, MAX_INTENTOS = 4, 10


def pistas(secreto, intento):
    return tuple("correcto" if s == i else "presente" if i in secreto else "ausente" for s, i in zip(secreto, intento))


def es_valido(intento):
    return len(intento) == LONGITUD and set(intento) <= set(SIMBOLOS) and len(set(intento)) == LONGITUD


def partida(secreto, propuestas, mostrar=True):
    usados = 0
    for intento in propuestas:
        if not es_valido(intento):
            if mostrar:
                print(f"  {intento!r} no es válido: hacen falta {LONGITUD} símbolos distintos de {SIMBOLOS}")
            continue
        usados += 1
        p = pistas(secreto, intento)
        if mostrar:
            print(f"  {usados:>2}. {intento} → {', '.join(p)}")
        if intento == secreto:
            return usados
        if usados == MAX_INTENTOS:
            break
    return None


resultado = partida("B3A1", ["A12B", "AAB1", "C3B1", "B3C", "B31A", "B3A1"])
print("Acertado en", resultado, "intentos")
assert resultado == 4

todos = ["".join(p) for p in permutations(SIMBOLOS, LONGITUD)]
random.seed(2)
for _ in range(500):
    secreto, intento = random.sample(todos, 2)
    p = pistas(secreto, intento)
    assert p.count("correcto") + p.count("presente") == len(set(secreto) & set(intento))
assert partida("123A", ["ABC1"] * 12, mostrar=False) is None

   1. A12B → presente, presente, ausente, presente
  'AAB1' no es válido: hacen falta 4 símbolos distintos de ABC123
   2. C3B1 → ausente, correcto, presente, correcto
  'B3C' no es válido: hacen falta 4 símbolos distintos de ABC123
   3. B31A → correcto, correcto, presente, presente
   4. B3A1 → correcto, correcto, correcto, correcto
Acertado en 4 intentos


## 3. Un jugador automático que siempre gana

Escribir un jugador que empiece con las $6 \cdot 5 \cdot 4 \cdot 3 = 360$ combinaciones posibles, proponga siempre la primera candidata y descarte después las que no darían las mismas pistas si fueran el secreto. Jugar contra los 360 secretos posibles y comprobar que siempre acierta en 10 intentos o menos.

In [3]:
from collections import Counter


def jugador_automatico(secreto, mostrar=False):
    candidatos = list(todos)
    for intento_n in range(1, MAX_INTENTOS + 1):
        intento = candidatos[0]
        p = pistas(secreto, intento)
        if mostrar:
            print(f"  {intento_n:>2}. {intento} → {', '.join(p)} (quedan {len(candidatos)})")
        if intento == secreto:
            return intento_n
        candidatos = [c for c in candidatos if pistas(c, intento) == p]
    return None


print("Partida contra el secreto 3C1B:")
jugador_automatico("3C1B", mostrar=True)

reparto = Counter(jugador_automatico(s) for s in todos)
print("\nIntentos necesarios para los 360 secretos:")
for intentos, veces in sorted(reparto.items()):
    print(f"  {intentos} intentos: {veces:>3} {'█' * (veces // 4)}")
print(f"Media: {sum(k * v for k, v in reparto.items()) / len(todos):.2f}")
assert None not in reparto and max(reparto) <= MAX_INTENTOS and reparto.total() == 360

Partida contra el secreto 3C1B:
   1. ABC1 → ausente, presente, presente, presente (quedan 360)
   2. BC12 → presente, correcto, correcto, ausente (quedan 22)
   3. 3C1B → correcto, correcto, correcto, correcto (quedan 1)

Intentos necesarios para los 360 secretos:
  1 intentos:   1 
  2 intentos:  67 ████████████████
  3 intentos: 236 ███████████████████████████████████████████████████████████
  4 intentos:  55 █████████████
  5 intentos:   1 
Media: 2.97


## 4. Ahorcado

Escribir `ahorcado(palabra, letras, max_fallos=6)`, que juegue una partida con una lista fija de letras propuestas: muestre tras cada una la palabra con las letras descubiertas y los fallos, avise de las letras repetidas o no válidas sin penalizar, y pare al descubrir la palabra o al llegar al máximo de fallos. Jugar una partida ganada y otra perdida.

In [4]:
def ahorcado(palabra, letras, max_fallos=6):
    palabra = palabra.upper()
    probadas, fallos = set(), 0
    for letra in map(str.upper, letras):
        if len(letra) != 1 or not letra.isalpha():
            print(f"  {letra!r}: no es una letra")
            continue
        if letra in probadas:
            print(f"  {letra}: ya la habías dicho")
            continue
        probadas.add(letra)
        if letra not in palabra:
            fallos += 1
        visible = " ".join(c if c in probadas else "_" for c in palabra)
        print(f"  {letra}: {visible}   fallos {fallos}/{max_fallos}")
        if set(palabra) <= probadas:
            return True, fallos
        if fallos == max_fallos:
            return False, fallos
    return False, fallos


print("TAJINASTE")
ganada, fallos = ahorcado("tajinaste", ["e", "a", "o", "s", "a", "t", "?", "r", "i", "n", "j"])
print("Ganada" if ganada else "Perdida", "con", fallos, "fallos\n")
assert ganada and fallos == 2

print("BARRANCO")
perdida = ahorcado("barranco", ["e", "i", "a", "u", "s", "t", "l", "p", "r"])
print("Ganada" if perdida[0] else "Perdida", "con", perdida[1], "fallos")
assert perdida == (False, 6)

TAJINASTE
  E: _ _ _ _ _ _ _ _ E   fallos 0/6
  A: _ A _ _ _ A _ _ E   fallos 0/6
  O: _ A _ _ _ A _ _ E   fallos 1/6
  S: _ A _ _ _ A S _ E   fallos 1/6
  A: ya la habías dicho
  T: T A _ _ _ A S T E   fallos 1/6
  '?': no es una letra
  R: T A _ _ _ A S T E   fallos 2/6
  I: T A _ I _ A S T E   fallos 2/6
  N: T A _ I N A S T E   fallos 2/6
  J: T A J I N A S T E   fallos 2/6
Ganada con 2 fallos

BARRANCO
  E: _ _ _ _ _ _ _ _   fallos 1/6
  I: _ _ _ _ _ _ _ _   fallos 2/6
  A: _ A _ _ A _ _ _   fallos 2/6
  U: _ A _ _ A _ _ _   fallos 3/6
  S: _ A _ _ A _ _ _   fallos 4/6
  T: _ A _ _ A _ _ _   fallos 5/6
  L: _ A _ _ A _ _ _   fallos 6/6
Perdida con 6 fallos


## 5. Combate por turnos

Crear una `dataclass` `Luchador` con nombre, vida máxima, ataque, defensa, agilidad (probabilidad de esquivar) y velocidad. En cada asalto ataca primero el más rápido: el daño es un entero al azar entre la mitad del ataque y el ataque, menos la defensa del rival (si la defensa lo anula, el golpe quita un 10 %), y el rival lo esquiva con su probabilidad de agilidad. Al final de cada asalto, los que siguen en pie recuperan 2 puntos de vida sin pasar del máximo. Escribir `combate(a, b, rng)`, que devuelva el ganador, y comprobar que la vida nunca sale de sus límites.

In [5]:
from dataclasses import dataclass, field


@dataclass
class Luchador:
    nombre: str
    vida_max: int
    ataque: int
    defensa: int
    agilidad: float
    velocidad: int
    vida: int = field(init=False)

    def __post_init__(self):
        self.vida = self.vida_max

    def recibe(self, atacante, rng):
        if rng.random() < self.agilidad:
            return 0
        golpe = rng.randint(atacante.ataque // 2, atacante.ataque)
        dano = golpe - self.defensa if golpe > self.defensa else max(1, golpe // 10)
        self.vida = max(0, self.vida - dano)
        return dano


def combate(a, b, rng, mostrar=True, max_asaltos=500):
    a.vida, b.vida = a.vida_max, b.vida_max
    orden = sorted([a, b], key=lambda l: l.velocidad, reverse=True)
    for asalto in range(1, max_asaltos + 1):
        for atacante, defensor in (orden, orden[::-1]):
            dano = defensor.recibe(atacante, rng)
            if mostrar and asalto <= 3:
                texto = f"quita {dano}" if dano else "falla: el rival esquiva"
                print(f"  Asalto {asalto}: {atacante.nombre} ataca y {texto} ({defensor.nombre}: {defensor.vida})")
            assert 0 <= defensor.vida <= defensor.vida_max
            if defensor.vida == 0:
                if mostrar:
                    print(f"  … {defensor.nombre} cae en el asalto {asalto}; gana {atacante.nombre} con {atacante.vida} de vida")
                return atacante
        for l in (a, b):
            l.vida = min(l.vida_max, l.vida + 2)
    return max((a, b), key=lambda l: l.vida)


tinguaro = Luchador("Tinguaro", 120, 30, 8, 0.15, 6)
guayarmina = Luchador("Guayarmina", 100, 34, 5, 0.30, 8)
ganador = combate(tinguaro, guayarmina, random.Random(5))
assert ganador.vida > 0 and min(tinguaro.vida, guayarmina.vida) == 0

victorias = Counter(combate(tinguaro, guayarmina, random.Random(s), mostrar=False).nombre for s in range(1000))
print("En 1000 combates:", dict(victorias))
assert victorias.total() == 1000

  Asalto 1: Guayarmina ataca y quita 20 (Tinguaro: 100)
  Asalto 1: Tinguaro ataca y quita 10 (Guayarmina: 90)
  Asalto 2: Guayarmina ataca y quita 16 (Tinguaro: 86)
  Asalto 2: Tinguaro ataca y quita 15 (Guayarmina: 77)
  Asalto 3: Guayarmina ataca y falla: el rival esquiva (Tinguaro: 88)
  Asalto 3: Tinguaro ataca y quita 17 (Guayarmina: 62)
  … Tinguaro cae en el asalto 11; gana Guayarmina con 13 de vida
En 1000 combates: {'Tinguaro': 410, 'Guayarmina': 590}


## 6. Torneo eliminatorio

Escribir `torneo(luchadores, rng)` para un número de luchadores que sea potencia de 2 (si no, `ValueError`): se emparejan al azar y en cada ronda pasan los ganadores hasta que queda un campeón. Cada combate empieza con la vida al máximo. Comprobar que se disputan $n - 1$ combates y que cada ronda tiene la mitad de luchadores que la anterior.

In [6]:
def torneo(luchadores, rng):
    n = len(luchadores)
    if n < 2 or n & (n - 1):
        raise ValueError(f"hacen falta 2, 4, 8, 16… luchadores, no {n}")
    ronda, combates, tamanos = list(luchadores), 0, []
    rng.shuffle(ronda)
    while len(ronda) > 1:
        tamanos.append(len(ronda))
        print(f"Ronda de {len(ronda)}:")
        siguiente = []
        for a, b in zip(ronda[::2], ronda[1::2]):
            g = combate(a, b, rng, mostrar=False)
            combates += 1
            print(f"  {a.nombre:<11} vs {b.nombre:<11} → {g.nombre}")
            siguiente.append(g)
        ronda = siguiente
    return ronda[0], combates, tamanos


rng = random.Random(6)
nombres = ["Tinguaro", "Guayarmina", "Bentejuí", "Doramas", "Acaymo", "Tibiabin", "Beneharo", "Arminda"]
plantel = [
    Luchador(n, rng.randint(90, 130), rng.randint(22, 36), rng.randint(2, 10), round(rng.uniform(0.05, 0.3), 2), rng.randint(1, 10))
    for n in nombres
]
for l in plantel:
    print(f"{l.nombre:<11} vida {l.vida_max:>3}  ataque {l.ataque}  defensa {l.defensa:>2}  agilidad {l.agilidad:.2f}  velocidad {l.velocidad:>2}")
campeon, combates, tamanos = torneo(plantel, rng)
print("Campeón:", campeon.nombre)
assert combates == len(plantel) - 1 and tamanos == [8, 4, 2] and campeon in plantel

try:
    torneo(plantel[:6], rng)
except ValueError as e:
    print("Error:", e)

Tinguaro    vida 126  ataque 35  defensa  3  agilidad 0.17  velocidad  5
Guayarmina  vida  92  ataque 22  defensa  4  agilidad 0.22  velocidad  8
Bentejuí    vida 113  ataque 27  defensa  2  agilidad 0.12  velocidad  4
Doramas     vida 116  ataque 36  defensa 10  agilidad 0.18  velocidad  2
Acaymo      vida 102  ataque 31  defensa 10  agilidad 0.22  velocidad  5
Tibiabin    vida 129  ataque 32  defensa  3  agilidad 0.26  velocidad  6
Beneharo    vida  95  ataque 27  defensa  8  agilidad 0.25  velocidad  8
Arminda     vida  96  ataque 34  defensa  5  agilidad 0.23  velocidad  5
Ronda de 8:
  Doramas     vs Beneharo    → Doramas
  Tibiabin    vs Bentejuí    → Tibiabin
  Arminda     vs Acaymo      → Acaymo
  Tinguaro    vs Guayarmina  → Tinguaro
Ronda de 4:
  Doramas     vs Tibiabin    → Doramas
  Acaymo      vs Tinguaro    → Tinguaro
Ronda de 2:
  Doramas     vs Tinguaro    → Doramas
Campeón: Doramas
Error: hacen falta 2, 4, 8, 16… luchadores, no 6


## 7. Batalla entre dos ejércitos

Cada tipo de unidad tiene un valor (honderos 1, lanceros 2, arqueros 3, jinetes 4, gigantes 5). Escribir `poder(ejercito)` para un ejército dado como diccionario `{unidad: cantidad}` y `batalla(norte, sur)`, que diga quién gana o si hay empate. Escribir también `refuerzo_minimo(perdedor, ganador, unidad)`, el menor número de unidades de un tipo que necesita el bando que pierde para ganar, y comprobarlo añadiendo unidades de una en una.

In [7]:
VALOR = {"honderos": 1, "lanceros": 2, "arqueros": 3, "jinetes": 4, "gigantes": 5}


def poder(ejercito):
    return sum(VALOR[u] * n for u, n in ejercito.items())


def batalla(norte, sur):
    pn, ps = poder(norte), poder(sur)
    if pn == ps:
        return f"Empate ({pn} a {ps})"
    return f"Gana el {'Norte' if pn > ps else 'Sur'} ({pn} a {ps})"


def refuerzo_minimo(perdedor, ganador, unidad):
    falta = poder(ganador) - poder(perdedor) + 1
    return max(0, -(-falta // VALOR[unidad]))


norte = {"honderos": 30, "lanceros": 25, "jinetes": 8}
sur = {"arqueros": 20, "gigantes": 6, "lanceros": 10, "honderos": 12}
print("Norte:", norte, "→", poder(norte))
print("Sur:  ", sur, "→", poder(sur))
print(batalla(norte, sur))
for unidad in VALOR:
    print(f"  El Norte necesitaría {refuerzo_minimo(norte, sur, unidad):>2} {unidad} más para ganar")

for unidad in VALOR:
    extra = 0
    while poder(norte | {unidad: norte.get(unidad, 0) + extra}) <= poder(sur):
        extra += 1
    assert extra == refuerzo_minimo(norte, sur, unidad)
assert batalla({"jinetes": 3}, {"lanceros": 6}) == "Empate (12 a 12)"

Norte: {'honderos': 30, 'lanceros': 25, 'jinetes': 8} → 112
Sur:   {'arqueros': 20, 'gigantes': 6, 'lanceros': 10, 'honderos': 12} → 122
Gana el Sur (112 a 122)
  El Norte necesitaría 11 honderos más para ganar
  El Norte necesitaría  6 lanceros más para ganar
  El Norte necesitaría  4 arqueros más para ganar
  El Norte necesitaría  3 jinetes más para ganar
  El Norte necesitaría  3 gigantes más para ganar


## 8. Reparto de anillos

Se reparten $n$ anillos entre un guardián y tres pueblos: el guardián recibe exactamente 1, el pueblo de la costa un número impar, el de la montaña un número primo y el del valle un número par mayor que cero. Escribir `repartos(n)` con todos los repartos válidos y `reparto_equilibrado(n)`, que elija el que menos diferencia deja entre el pueblo que más recibe y el que menos, o lance `ValueError` si no hay ninguno. Comprobar con los repartos de 1 a 60 que hay solución si y solo si $n \ge 6$ (el mínimo es $1 + 1 + 2 + 2$, y por paridad siempre se puede completar a partir de ahí).

In [8]:
def es_primo(k):
    return k >= 2 and all(k % d for d in range(2, int(k**0.5) + 1))


def repartos(n):
    resto = n - 1
    return [
        (costa, montana, resto - costa - montana)
        for costa in range(1, resto + 1, 2)
        for montana in range(2, resto - costa + 1)
        if es_primo(montana) and (valle := resto - costa - montana) > 0 and valle % 2 == 0
    ]


def reparto_equilibrado(n):
    opciones = repartos(n)
    if not opciones:
        raise ValueError(f"no se pueden repartir {n} anillos con estas reglas")
    return min(opciones, key=lambda r: (max(r) - min(r), r))


for n in (5, 6, 7, 20, 33):
    try:
        costa, montana, valle = reparto_equilibrado(n)
        print(f"{n:>2} anillos: guardián 1, costa {costa}, montaña {montana}, valle {valle}  ({len(repartos(n))} repartos posibles)")
    except ValueError as e:
        print(f"{n:>2} anillos: {e}")

for n in range(1, 61):
    for costa, montana, valle in repartos(n):
        assert 1 + costa + montana + valle == n and costa % 2 == 1 and es_primo(montana) and valle % 2 == 0 < valle
    assert bool(repartos(n)) == (n >= 6)

 5 anillos: no se pueden repartir 5 anillos con estas reglas
 6 anillos: guardián 1, costa 1, montaña 2, valle 2  (1 repartos posibles)
 7 anillos: guardián 1, costa 1, montaña 3, valle 2  (1 repartos posibles)
20 anillos: guardián 1, costa 9, montaña 2, valle 8  (8 repartos posibles)
33 anillos: guardián 1, costa 11, montaña 11, valle 10  (76 repartos posibles)


## 9. Carrera de coches

Dos coches corren en carriles de 15 casillas con 3 obstáculos colocados al azar en cada uno (ni en la salida ni en la meta). En cada turno, cada coche avanza de 1 a 3 casillas al azar sin pasar de la meta; si cae en un obstáculo, pierde el turno siguiente. Gana el primero en llegar, y si llegan en el mismo turno es empate. Dibujar los carriles cada turno y comprobar que las posiciones nunca retroceden y que el coche penalizado no se mueve en su turno de castigo.

In [9]:
META = 14


def carrera(rng, mostrar=True):
    carriles = [set(rng.sample(range(1, META), 3)) for _ in range(2)]
    posiciones, castigo, historial = [0, 0], [False, False], [(0, 0)]
    while META not in posiciones:
        for k in range(2):
            if castigo[k]:
                castigo[k] = False
                continue
            posiciones[k] = min(META, posiciones[k] + rng.randint(1, 3))
            castigo[k] = posiciones[k] in carriles[k]
        historial.append(tuple(posiciones))
        for k in range(2) if mostrar else ():
            carril = ["*" if i in carriles[k] else "_" for i in range(META + 1)]
            carril[posiciones[k]] = "AB"[k]
            print("".join(carril) + "|", end="   " if k == 0 else "\n")
    llegan = [coche for coche, p in zip("AB", posiciones) if p == META]
    return ("empate" if len(llegan) == 2 else llegan[0]), historial, carriles


ganador, historial, carriles = carrera(random.Random(9))
print("Resultado:", ganador)

for semilla in range(300):
    ganador, historial, carriles = carrera(random.Random(semilla), mostrar=False)
    for k in range(2):
        pos = [h[k] for h in historial]
        assert all(0 <= p <= META for p in pos) and pos == sorted(pos)
        for t in range(1, len(pos) - 1):
            if pos[t] in carriles[k] and pos[t] != pos[t - 1]:
                assert pos[t + 1] == pos[t]
    assert ganador == "empate" or historial[-1]["AB".index(ganador)] == META

___A__*_*_*____|   _B_*_*______*__|
_____A*_*_*____|   ___*B*______*__|
______*A*_*____|   ___*_*_B____*__|
______*_A_*____|   ___*_*___B__*__|
______*_A_*____|   ___*_*______B__|
______*_*_*A___|   ___*_*______B__|
______*_*_*___A|   ___*_*______*B_|
Resultado: A


## 10. Simulador de clima

Simular $N$ días del tiempo de un pueblo con estas reglas inventadas: se empieza con 20 °C y un 20 % de probabilidad de lluvia; cada día la temperatura sube 2 °C con probabilidad 0,1 y baja 2 °C con probabilidad 0,1; si llueve, al día siguiente baja 1 °C más, y si no llueve y está por debajo de 20 °C, sube 1 °C; un día por encima de 22 °C sube 10 puntos la probabilidad de lluvia del siguiente y uno por debajo de 18 °C la baja 10 (siempre entre 0 y 100). Escribir `simular_clima(dias, rng)` y mostrar la temperatura máxima, la mínima y los días de lluvia.

In [10]:
def simular_clima(dias, rng):
    temperatura, prob_lluvia = 20, 20
    registro = []
    for dia in range(1, dias + 1):
        azar = rng.random()
        temperatura += 2 if azar < 0.1 else -2 if azar < 0.2 else 0
        llueve = rng.random() * 100 < prob_lluvia
        registro.append((dia, temperatura, prob_lluvia, llueve))
        if llueve:
            temperatura -= 1
        elif temperatura < 20:
            temperatura += 1
        if temperatura > 22:
            prob_lluvia = min(100, prob_lluvia + 10)
        elif temperatura < 18:
            prob_lluvia = max(0, prob_lluvia - 10)
    return registro


registro = simular_clima(90, random.Random(10))
print("Día  °C  P(lluvia)")
for dia, t, p, llueve in registro[:12]:
    print(f"{dia:>3} {t:>3}  {p:>5} %  {'lluvia' if llueve else ''}")
temperaturas = [t for _, t, _, _ in registro]
dias_lluvia = sum(ll for *_, ll in registro)
print(f"…\nEn {len(registro)} días: máxima {max(temperaturas)} °C, mínima {min(temperaturas)} °C, {dias_lluvia} días de lluvia")

for semilla in range(200):
    r = simular_clima(365, random.Random(semilla))
    assert all(0 <= p <= 100 for _, _, p, _ in r)
    for (_, t0, _, ll), (_, t1, _, _) in zip(r, r[1:]):
        assert t1 - t0 + (1 if ll else -1 if t0 < 20 else 0) in (-2, 0, 2)

Día  °C  P(lluvia)
  1  20     20 %  
  2  20     20 %  
  3  20     20 %  
  4  20     20 %  lluvia
  5  19     20 %  
  6  20     20 %  
  7  20     20 %  lluvia
  8  19     20 %  
  9  20     20 %  
 10  20     20 %  
 11  20     20 %  
 12  18     20 %  
…
En 90 días: máxima 25 °C, mínima 15 °C, 24 días de lluvia


## 11. ¿A qué casa perteneces?

Un cuestionario de 10 preguntas reparte a la gente entre cuatro casas: Drago, Pino, Cardón y Tabaiba. Cada respuesta da puntos a una casa (las preguntas de personalidad valen 2 y las demás 1). Escribir `asignar_casa(respuestas, rng)`, que sume los puntos de unas respuestas fijas y, si hay empate en cabeza, elija al azar entre las empatadas. Comprobar que los puntos repartidos suman lo que valen las preguntas y que, con distintas semillas, cualquiera de las casas empatadas puede salir.

In [11]:
CASAS = ["Drago", "Pino", "Cardón", "Tabaiba"]
PREGUNTAS = [
    ("¿Qué plan prefieres un domingo?", 1, {"subir a la cumbre": "Pino", "ir a la playa": "Tabaiba",
                                            "leer bajo un árbol": "Drago", "quedar con amigos": "Cardón"}),
    ("¿Qué haces si un amigo tiene un problema?", 2, {"lo escucho": "Drago", "busco una solución": "Cardón",
                                                       "lo animo con un plan": "Tabaiba", "le doy espacio": "Pino"}),
    ("Elige un momento del día", 1, {"amanecer": "Pino", "mediodía": "Cardón", "atardecer": "Tabaiba", "noche": "Drago"}),
    ("¿Cómo afrontas un examen?", 2, {"con un plan": "Cardón", "con calma": "Drago",
                                      "en grupo": "Tabaiba", "a última hora": "Pino"}),
    ("Elige un paisaje", 1, {"bosque de laurisilva": "Pino", "malpaís": "Cardón", "dunas": "Tabaiba", "barranco": "Drago"}),
    ("¿Qué valoras más en la gente?", 2, {"la lealtad": "Drago", "el valor": "Cardón",
                                          "la alegría": "Tabaiba", "la curiosidad": "Pino"}),
    ("Elige una comida", 1, {"papas arrugadas": "Cardón", "potaje": "Drago", "pescado": "Tabaiba", "fruta": "Pino"}),
    ("¿Qué te gustaría aprender?", 1, {"astronomía": "Pino", "música": "Tabaiba", "historia": "Drago", "a construir cosas": "Cardón"}),
    ("En un trabajo en equipo, tú…", 2, {"organizo": "Cardón", "propongo ideas": "Pino",
                                         "mantengo el buen ambiente": "Tabaiba", "reviso los detalles": "Drago"}),
    ("Elige un animal", 1, {"lagarto": "Cardón", "pinzón azul": "Pino", "pardela": "Tabaiba", "lechuza": "Drago"}),
]


def puntuar(respuestas):
    puntos = dict.fromkeys(CASAS, 0)
    for (_, valor, opciones), r in zip(PREGUNTAS, respuestas, strict=True):
        puntos[opciones[r]] += valor
    return puntos


def asignar_casa(respuestas, rng):
    puntos = puntuar(respuestas)
    maximo = max(puntos.values())
    empatadas = [c for c, p in puntos.items() if p == maximo]
    return rng.choice(empatadas), puntos, empatadas


personas = {
    "Idaira": ["subir a la cumbre", "le doy espacio", "amanecer", "con calma", "bosque de laurisilva",
               "la curiosidad", "fruta", "astronomía", "propongo ideas", "lechuza"],
    "Airam": ["quedar con amigos", "lo escucho", "noche", "con un plan", "barranco",
              "la lealtad", "papas arrugadas", "historia", "organizo", "lagarto"],
}
rng = random.Random(11)
for nombre, respuestas in personas.items():
    casa, puntos, empatadas = asignar_casa(respuestas, rng)
    detalle = ", ".join(f"{c} {p}" for c, p in puntos.items())
    print(f"{nombre:<7} {detalle:<40} → {casa}{' (desempate entre ' + ' y '.join(empatadas) + ')' if len(empatadas) > 1 else ''}")
    assert sum(puntos.values()) == sum(v for _, v, _ in PREGUNTAS) == 14
    assert casa in empatadas

salidas = {asignar_casa(personas["Airam"], random.Random(s))[0] for s in range(50)}
assert salidas == set(asignar_casa(personas["Airam"], rng)[2]) == {"Drago", "Cardón"}
assert asignar_casa(personas["Idaira"], rng)[0] == "Pino"

Idaira  Drago 3, Pino 11, Cardón 0, Tabaiba 0    → Pino
Airam   Drago 7, Pino 0, Cardón 7, Tabaiba 0     → Cardón (desempate entre Drago y Cardón)
